# Markowitz Mean-Variance Optimization (MPT)

MPT answers one question: **given these assets, what is the best way to combine them?**

A robo-advisor uses this to build portfolios. It finds the weights that give the best
return for a given level of risk — or the best risk-adjusted return overall.

This notebook builds three portfolios and compares them:
- **Max Sharpe** — best return per unit of risk (what robo-advisors aim for)
- **Min Variance** — lowest possible risk, regardless of return
- **Risk Profile** — matches a specific investor risk tolerance (as described in the paper)
- **Equal Weight** — simple baseline

Assets: `BND, GLD, TIP, TLT, VEU, VGT`

In [ ]:
# ── Cell 1: Imports & Config ─────────────────────────────────────────────────
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.optimize as sco

warnings.filterwarnings("ignore")
np.random.seed(42)

# Load config from markets.json
with open("../../markets.json") as f:
    config = json.load(f)

BASE = Path("../../data")
name = list(config["markets"].keys())[0]

RISK_FREE_ANNUAL  = 0.0351
RISK_FREE_MONTHLY = (1 + RISK_FREE_ANNUAL) ** (1/12) - 1

# Crisis periods used in charts
CRISES = [
    ("2008-09", "2009-06", "GFC"),
    ("2020-02", "2020-04", "COVID"),
    ("2022-01", "2022-12", "Rate Hikes"),
]

PALETTE = ["#028090", "#F6C90E", "#E53E3E", "#38A169", "#6B46C1", "#C05621"]
plt.rcParams.update({
    "figure.facecolor" : "#F4F7FB",
    "axes.facecolor"   : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.4,
    "grid.color"       : "#CBD5E0",
    "font.family"      : "sans-serif",
})

print(f"Market    : {name}")
print(f"Risk-free : {RISK_FREE_ANNUAL*100:.2f}% annual")

In [ ]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
rets = pd.read_csv(
    BASE / f"{name}_returns_monthly.csv", index_col=0, parse_dates=True
)

ASSETS = list(rets.columns)
N      = len(ASSETS)

print(f"Period : {rets.index[0].date()} -> {rets.index[-1].date()}")
print(f"Assets : {ASSETS}")

In [ ]:
# ── Cell 3: Build the MPT Inputs ─────────────────────────────────────────────
#
# MPT needs two things:
#   mu  = expected return for each asset (annualized)
#   cov = how assets move together (annualized covariance matrix)
#
# We estimate both from historical monthly data, then scale up to annual.

mu      = rets.mean() * 12        # annualized expected return
cov     = rets.cov()  * 12        # annualized covariance matrix
mu_arr  = mu.values
cov_arr = cov.values

# Quick look at what we're working with
summary = pd.DataFrame({
    "Ann. Return %" : (mu * 100).round(2),
    "Ann. Vol %"    : (rets.std() * np.sqrt(12) * 100).round(2),
})
print(summary.to_string())

## 1. Find the Optimal Portfolios

In [ ]:
# ── Cell 4: Solve for Optimal Portfolios ─────────────────────────────────────
#
# We use scipy to solve three optimization problems.
# All portfolios: long-only (no short selling), fully invested (weights sum to 1).
#
# MAX SHARPE
#   Find weights that maximize (return - risk_free) / volatility.
#   This is the portfolio a robo-advisor targets for a "balanced" investor.
#
# MIN VARIANCE
#   Find weights that minimize total portfolio volatility.
#   This is what a robo-advisor offers a very conservative investor.
#
# RISK PROFILE (from the paper)
#   Fix a target volatility matching the investor's risk tolerance.
#   Maximize return subject to that constraint.
#   This is how Finizens and Indexa Capital actually work.

constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
bounds      = [(0.0, 1.0)] * N
w0          = np.ones(N) / N

# --- Max Sharpe ---
def neg_sharpe(w):
    r = w @ mu_arr
    v = np.sqrt(w @ cov_arr @ w)
    return -(r - RISK_FREE_ANNUAL) / v

w_sharpe = sco.minimize(
    neg_sharpe, w0, method="SLSQP",
    bounds=bounds, constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 1000}
).x
w_sharpe = pd.Series(w_sharpe, index=ASSETS)

# --- Min Variance ---
w_minvar = sco.minimize(
    lambda w: w @ cov_arr @ w, w0, method="SLSQP",
    bounds=bounds, constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 1000}
).x
w_minvar = pd.Series(w_minvar, index=ASSETS)

# --- Risk Profile (medium risk = annualized vol target of 8%) ---
# Change VOL_TARGET to match any investor risk profile
VOL_TARGET = 0.08

risk_constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1},
    {"type": "eq", "fun": lambda w: np.sqrt(w @ cov_arr @ w) - VOL_TARGET},
]
res_profile = sco.minimize(
    lambda w: -(w @ mu_arr), w0, method="SLSQP",
    bounds=bounds, constraints=risk_constraints,
    options={"ftol": 1e-12, "maxiter": 1000}
)
w_profile = pd.Series(res_profile.x, index=ASSETS)

# --- Equal Weight ---
w_equal = pd.Series(1 / N, index=ASSETS)

# --- Print results ---
def portfolio_stats(w, label):
    r = w.values @ mu_arr
    v = np.sqrt(w.values @ cov_arr @ w.values)
    s = (r - RISK_FREE_ANNUAL) / v
    return {"Portfolio": label,
            "Return %": round(r*100, 2),
            "Vol %":    round(v*100, 2),
            "Sharpe":   round(s, 3)}

metrics = pd.DataFrame([
    portfolio_stats(w_sharpe,  "Max Sharpe"),
    portfolio_stats(w_minvar,  "Min Variance"),
    portfolio_stats(w_profile, f"Risk Profile ({VOL_TARGET*100:.0f}% vol target)"),
    portfolio_stats(w_equal,   "Equal Weight"),
]).set_index("Portfolio")

print("── Portfolio Metrics (annualized) ──")
print(metrics.to_string())

## 2. Efficient Frontier

In [ ]:
# ── Cell 5: Efficient Frontier ────────────────────────────────────────────────
#
# Simulate 10,000 random portfolios and plot each as a dot.
# The upper-left edge of the cloud is the efficient frontier.
# Color shows the Sharpe ratio — green = better risk-adjusted return.
# The three optimal portfolios are marked on top.

N_SIM = 10_000
sim_r, sim_v, sim_s = np.zeros(N_SIM), np.zeros(N_SIM), np.zeros(N_SIM)

for i in range(N_SIM):
    w = np.random.dirichlet(np.ones(N))
    r = w @ mu_arr
    v = np.sqrt(w @ cov_arr @ w)
    sim_r[i] = r * 100
    sim_v[i] = v * 100
    sim_s[i] = (r - RISK_FREE_ANNUAL) / v

fig, ax = plt.subplots(figsize=(13, 7))

# Random portfolio cloud
sc = ax.scatter(sim_v, sim_r, c=sim_s, cmap="RdYlGn",
                alpha=0.3, s=8, zorder=1)
plt.colorbar(sc, ax=ax, label="Sharpe Ratio")

# Individual assets
for asset, color in zip(ASSETS, PALETTE):
    i  = ASSETS.index(asset)
    av = np.sqrt(cov_arr[i, i]) * 100
    ar = mu_arr[i] * 100
    ax.scatter(av, ar, s=100, color=color, zorder=5,
               edgecolors="white", linewidths=1)
    ax.annotate(asset, (av, ar), xytext=(6, 4),
                textcoords="offset points",
                fontsize=9, color=color, fontweight="bold")

# Key portfolios
key_ports = [
    (w_sharpe,  "Max Sharpe",   "#1A202C", "*",  300),
    (w_minvar,  "Min Variance", "#6B46C1", "D",  180),
    (w_profile, f"Risk Profile ({VOL_TARGET*100:.0f}% vol)", "#E53E3E", "^", 180),
    (w_equal,   "Equal Weight", "#028090", "o",  160),
]
for w, label, color, marker, size in key_ports:
    wv = w.values
    rv = wv @ mu_arr * 100
    vv = np.sqrt(wv @ cov_arr @ wv) * 100
    ax.scatter(vv, rv, s=size, color=color, marker=marker,
               zorder=6, edgecolors="white", linewidths=1.5,
               label=f"{label}  ({rv:.1f}%, {vv:.1f}%vol)")

# Capital Market Line from Max Sharpe
wv = w_sharpe.values
vv = np.sqrt(wv @ cov_arr @ wv) * 100
rv = wv @ mu_arr * 100
cml_x = np.linspace(0, vv * 1.4, 100)
cml_y = RISK_FREE_ANNUAL * 100 + (rv - RISK_FREE_ANNUAL*100) / vv * cml_x
ax.plot(cml_x, cml_y, color="#1A202C", linewidth=1.2,
        linestyle="--", alpha=0.5, label="Capital Market Line")

ax.set_xlabel("Annualized Volatility (%)", fontsize=11)
ax.set_ylabel("Annualized Return (%)", fontsize=11)
ax.set_title("Efficient Frontier — Markowitz MPT",
             fontsize=14, fontweight="bold", pad=14)
ax.legend(fontsize=9, loc="lower right")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

## 3. What Does Each Portfolio Actually Hold?

In [ ]:
# ── Cell 6: Weight Comparison ─────────────────────────────────────────────────
#
# MPT often concentrates heavily in one or two assets and zeroes out the rest.
# This is called a "corner solution" — a known weakness of the model.
# Compare to Equal Weight to see how extreme the concentration is.

weight_df = pd.DataFrame({
    "Max Sharpe"   : w_sharpe * 100,
    "Min Variance" : w_minvar * 100,
    f"Risk Profile\n({VOL_TARGET*100:.0f}% vol)" : w_profile * 100,
    "Equal Weight" : w_equal  * 100,
})

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
colors_map = ["#1A202C", "#6B46C1", "#E53E3E", "#028090"]

for ax, col, color in zip(axes, weight_df.columns, colors_map):
    vals = weight_df[col]
    bar_colors = [color if v > 0.5 else "#E2E8F0" for v in vals]
    ax.bar(ASSETS, vals, color=bar_colors, alpha=0.9, width=0.6)
    for asset, val in zip(ASSETS, vals):
        if val > 0.5:
            ax.text(ASSETS.index(asset), val + 0.8,
                    f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")
    ax.set_title(col, fontweight="bold", fontsize=11)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Weight (%)" if ax == axes[0] else "")

fig.suptitle("Portfolio Weights — What Does Each Strategy Hold?",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(weight_df.round(1).to_string())

## 4. Historical Performance

In [ ]:
# ── Cell 7: Cumulative Returns & Drawdowns ────────────────────────────────────
#
# Apply each set of weights to the actual historical returns.
#
# Important: these weights are calculated from the full history.
# That means we are looking backwards — this will overstate real performance.
# The rolling simulation in the next cell is the honest version.

sharpe_ret  = (rets * w_sharpe ).sum(axis=1)
minvar_ret  = (rets * w_minvar ).sum(axis=1)
profile_ret = (rets * w_profile).sum(axis=1)
equal_ret   = (rets * w_equal  ).sum(axis=1)

def to_pct(ret_series):
    return ((1 + ret_series).cumprod() - 1) * 100

def drawdown(ret_series):
    cum = (1 + ret_series).cumprod()
    return (cum - cum.cummax()) / cum.cummax() * 100

def shade_crises(ax):
    for start, end, label in CRISES:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=0.08, color="red", zorder=0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)

# Cumulative returns
ax1.plot(to_pct(sharpe_ret),  color="#1A202C",  lw=2,   label="Max Sharpe")
ax1.plot(to_pct(minvar_ret),  color="#6B46C1",  lw=2,   label="Min Variance",  ls="--")
ax1.plot(to_pct(profile_ret), color="#E53E3E",  lw=2,   label=f"Risk Profile ({VOL_TARGET*100:.0f}%vol)", ls="-.")
ax1.plot(to_pct(equal_ret),   color="#028090",  lw=1.6, label="Equal Weight",  ls=":")
ax1.axhline(0, color="#8896A5", lw=0.8, ls="--")
shade_crises(ax1)
ax1.set_title("Cumulative Return", fontsize=13, fontweight="bold")
ax1.set_ylabel("Return (%)")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
ax1.legend(fontsize=10)

# Drawdowns
ax2.fill_between(rets.index, drawdown(sharpe_ret),  0, alpha=0.2, color="#1A202C")
ax2.fill_between(rets.index, drawdown(minvar_ret),  0, alpha=0.2, color="#6B46C1")
ax2.plot(drawdown(sharpe_ret),  color="#1A202C", lw=1.8, label=f"Max Sharpe   (max: {drawdown(sharpe_ret).min():.1f}%)")
ax2.plot(drawdown(minvar_ret),  color="#6B46C1", lw=1.8, label=f"Min Variance (max: {drawdown(minvar_ret).min():.1f}%)", ls="--")
ax2.plot(drawdown(profile_ret), color="#E53E3E", lw=1.8, label=f"Risk Profile (max: {drawdown(profile_ret).min():.1f}%)", ls="-.")
ax2.plot(drawdown(equal_ret),   color="#028090", lw=1.4, label=f"Equal Weight (max: {drawdown(equal_ret).min():.1f}%)",   ls=":")
ax2.axhline(0, color="#8896A5", lw=0.8)
shade_crises(ax2)
for start, _, label in CRISES:
    ax2.text(pd.Timestamp(start), -1.5, label, fontsize=8, color="#E53E3E")
ax2.set_title("Drawdown", fontsize=13, fontweight="bold")
ax2.set_ylabel("Drawdown (%)")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 5. Rolling Rebalancing — The Honest Test

A robo-advisor does not use future data to pick weights.
It re-optimizes every month using only the past 3 years of data,
then applies those weights to the *next* month.

This is a realistic simulation of what a real robo-advisor does.
The gap between this and the static performance above tells you
how much of MPT's edge is real vs. look-ahead bias.

In [ ]:
# ── Cell 8: Rolling Rebalancing Simulation ────────────────────────────────────

LOOKBACK = 36    # use 3 years of past data to optimize

rolling_weights = []
rolling_dates   = []
rolling_returns = []

for i in range(LOOKBACK, len(rets)):
    window  = rets.iloc[i - LOOKBACK : i]
    mu_w    = window.mean().values * 12
    cov_w   = window.cov().values  * 12

    def neg_sharpe_w(w):
        r = w @ mu_w
        v = np.sqrt(w @ cov_w @ w)
        return -(r - RISK_FREE_ANNUAL) / v if v > 1e-8 else 0

    try:
        res   = sco.minimize(neg_sharpe_w, np.ones(N)/N, method="SLSQP",
                             bounds=[(0,1)]*N,
                             constraints={"type":"eq","fun":lambda w: np.sum(w)-1},
                             options={"ftol":1e-10,"maxiter":500})
        w_opt = res.x
    except Exception:
        w_opt = np.ones(N) / N

    rolling_weights.append(w_opt)
    rolling_dates.append(rets.index[i])
    rolling_returns.append(w_opt @ rets.iloc[i].values)

rolling_w_df = pd.DataFrame(rolling_weights, index=rolling_dates, columns=ASSETS)
rolling_ret  = pd.Series(rolling_returns,    index=rolling_dates)

# --- Plot 1: How weights change over time ---
fig, ax = plt.subplots(figsize=(13, 5))
ax.stackplot(rolling_w_df.index,
             [rolling_w_df[c] * 100 for c in ASSETS],
             labels=ASSETS, colors=PALETTE, alpha=0.85)
for start, end, label in CRISES:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.10, color="white", zorder=2)
    ax.text(pd.Timestamp(start), 103, label, fontsize=7.5, color="#E53E3E")
ax.set_title(f"Rolling MPT — Weight Changes Over Time ({LOOKBACK}-month window)",
             fontsize=13, fontweight="bold", pad=14)
ax.set_ylabel("Weight (%)")
ax.set_ylim(0, 100)
ax.legend(loc="upper left", fontsize=9, ncol=N)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

# --- Plot 2: Rolling vs static vs equal weight ---
idx = rolling_ret.index
rolling_growth = to_pct(rolling_ret)
static_growth  = to_pct(sharpe_ret.loc[idx])
equal_growth   = to_pct(equal_ret.loc[idx])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(rolling_growth, color="#E53E3E",  lw=2,   label="Rolling MPT (realistic)")
ax.plot(static_growth,  color="#1A202C",  lw=1.8, label="Static Max Sharpe (in-sample bias)", ls="--")
ax.plot(equal_growth,   color="#028090",  lw=1.4, label="Equal Weight", ls=":")
ax.axhline(0, color="#8896A5", lw=0.8, ls="--")
shade_crises(ax)
ax.set_title("Rolling MPT vs Static MPT vs Equal Weight",
             fontsize=13, fontweight="bold", pad=14)
ax.set_ylabel("Cumulative Return (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Rolling MPT final : {rolling_growth.iloc[-1]:.1f}%")
print(f"Static Max Sharpe : {static_growth.iloc[-1]:.1f}%")
print(f"Equal Weight      : {equal_growth.iloc[-1]:.1f}%")

## 6. Performance Summary

In [ ]:
# ── Cell 9: Performance Summary ──────────────────────────────────────────────
# All metrics annualized from monthly returns.
# Sharpe uses the correct monthly risk-free rate.
# All series aligned to the rolling window period for a fair comparison.

def performance_stats(ret_series, label):
    ann_return = ret_series.mean() * 12 * 100
    ann_vol    = ret_series.std()  * np.sqrt(12) * 100
    sharpe     = (ret_series.mean() - RISK_FREE_MONTHLY) * 12 / \
                 (ret_series.std() * np.sqrt(12))
    cum        = (1 + ret_series).cumprod()
    max_dd     = ((cum - cum.cummax()) / cum.cummax()).min() * 100

    return pd.Series({
        "Ann. Return %"   : round(ann_return, 2),
        "Ann. Vol %"      : round(ann_vol,    2),
        "Sharpe"          : round(sharpe,     3),
        "Max Drawdown %"  : round(max_dd,     2),
    }, name=label)

idx = rolling_ret.index

summary = pd.DataFrame([
    performance_stats(sharpe_ret.loc[idx],   "Max Sharpe (static)"),
    performance_stats(minvar_ret.loc[idx],   "Min Variance (static)"),
    performance_stats(profile_ret.loc[idx],  f"Risk Profile {VOL_TARGET*100:.0f}% (static)"),
    performance_stats(rolling_ret,           "Max Sharpe (rolling)"),
    performance_stats(equal_ret.loc[idx],    "Equal Weight"),
])

print("── Performance Summary ──")
print(summary.to_string())

# Visual table
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis("off")
col_labels = ["Portfolio"] + list(summary.columns)
row_data   = [[idx] + [str(v) for v in summary.loc[idx]] for idx in summary.index]
table = ax.table(cellText=row_data, colLabels=col_labels,
                 cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.0)
for j in range(len(col_labels)):
    table[0, j].set_facecolor("#1E2761")
    table[0, j].set_text_props(color="white", fontweight="bold")
row_colors = ["#F8F9FA", "#F0F4FF", "#FFF5F0", "#FFF0F0", "#F0FFF4"]
for i, rc in enumerate(row_colors):
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(rc)
ax.set_title("MPT Performance Summary", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

## Key Takeaways

**MPT works — but with a catch**
The Max Sharpe portfolio sits exactly where it should on the efficient frontier,
and the static in-sample Sharpe looks impressive. But static = cheating.
It used future data to pick the weights.

**The rolling version is the honest test**
When MPT re-optimizes monthly using only past data (like a real robo-advisor),
performance drops significantly. The gap between static and rolling is the
"complexity premium" — what you give up by not having a crystal ball.

**Concentration is a problem**
MPT often puts 70-90% into one or two assets and zeros the rest.
This makes portfolios fragile. Risk Parity and other strategies fix this.

**The Risk Profile approach is how robo-advisors actually work**
Rather than maximizing Sharpe for everyone, a robo-advisor first asks
the investor how much risk they can tolerate, then finds the best portfolio
for that specific volatility target. Change `VOL_TARGET` in Cell 4 to
simulate different investor profiles.